# 01. Evaluation protocol: random split, spatial CV, dataset holdout, JECAM

## Setup

In [ ]:
from pathlib import Path
import json
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").is_file():
    if root == root.parent:
        raise RuntimeError("Open this notebook inside the project repository")
    root = root.parent
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
if str(root) not in sys.path:
    sys.path.insert(0, str(root))  # model_aza

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from model_aza.pilots import climate, clustering, data, earth_engine, evaluation, features
from model_aza.training import tuning
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
AREA = "pilots"


def show_and_save(figure, run, name):
    figure.tight_layout()
    figure.savefig(run / f"{name}.png", dpi=150)
    display(figure)
    plt.close(figure)


def repeated(name, X, rows, ok, seeds=(1, 2, 3), params=evaluation.PARAMS):
    """Mean and SD over random-forest seeds of every evaluation view."""
    summaries, details = [], []
    for seed in seeds:
        summary, holdout = evaluation.evaluate(name, X, rows, ok, params, seed)
        summaries.append({**summary, "seed": seed})
        details.extend({**r, "seed": seed} for r in holdout)
    return summaries, details


def summarize(summaries):
    table = pd.DataFrame(summaries)
    metrics = ["random_split_macro_f1", "spatial_cv_macro_f1", "dataset_holdout_macro_f1", "jecam_macro_f1",
               "jecam_carrot_f1", "jecam_cabbage_f1", "jecam_potato_f1"]
    grouped = table.groupby("feature_set", sort=False)
    out = grouped[metrics].agg(["mean", "std"]).round(3)
    out.insert(0, ("pixels_used", ""), grouped["pixels_used"].first())
    return out

## Inputs

In [ ]:
subset_run = ""  # Optional pipeline_outputs/pilots/subset_<id>; blank uses the latest
subset_run = Path(subset_run) if subset_run else tuning.latest_run(root, "subset", area=AREA)
rows, arrays = data.load(subset_run)
cube = features.stack(arrays["s2_values"])
valid = arrays["s2_valid"]
print("Subset:", subset_run, "| pixels:", len(rows))

## Baseline: fixed May-August interval predictors

In [ ]:
run = tuning.new_run(root, "protocol", subset_run / "meta.json", area=AREA)
X, ok, columns = features.fixed_interval(cube, valid)
summaries, details = repeated("fixed_may_aug", X, rows, ok)
tuning.save_records(run, "summary", summaries)
tuning.save_records(run, "dataset_holdout", details)
display(summarize(summaries))

## Gap between views

In [ ]:
table = pd.DataFrame(summaries)
views = ["random_split_macro_f1", "spatial_cv_macro_f1", "dataset_holdout_macro_f1", "jecam_macro_f1"]
gap = table[views].mean().rename("mean_macro_f1").to_frame()
gap["minus_random_split"] = gap.mean_macro_f1 - gap.mean_macro_f1.iloc[0]
display(gap.round(3))
display(pd.DataFrame(details).groupby("held_out_source")[["macro_f1_scored_crops", "carrot_recall", "potato_recall"]].mean().round(3))

## Figure

In [ ]:
figure, axis = plt.subplots(figsize=(8, 4))
axis.bar(range(len(views)), table[views].mean(), yerr=table[views].std(), capsize=4)
axis.set_xticks(range(len(views)), ["random split", "spatial CV", "dataset holdout", "JECAM test"])
axis.set_ylabel("Parcel macro-F1")
axis.set_ylim(0, 1)
show_and_save(figure, run, "evaluation_views")
print("Run folder:", run)